# 06 · Cerebellum: rejuvenating interventions vs the aging signal
Do dietary restriction (**aDR**) and young plasma (**YMP**) push cerebellar gene expression back towards the young state?

Everything is computed for **both** quantifications (kallisto = primary, STAR = robustness check) through the same DESeq2 code (`run_dds`).

| Analysis | Question |
|---|---|
| 2 | Which genes define aging (21 vs 3 months)? |
| 3 | Do intervention DEGs overlap aging DEGs, and in which direction? |
| 4 | Does an aging-signature score go down (rejuvenation) or up after treatment? |
| 5 | Which pathways (GSEA) change with aging, and are they reversed? |
| 6 | The paper's named gene sets in our data |

**Caveats that apply to every result:** 3–5 mice per group; interventions at 19 months (aDR: females, YMP: males) vs aging at 21 months (mixed sex), in different cohorts; p-values are indicative, not confirmatory. Signature scores are z-scored within each pair of groups, so cohort batch effects do not enter.

In [ ]:
source('../R/helpers.R')
samples <- load_samples()
x <- load_txi(rownames(samples)); genes <- x$genes
counts <- list(kallisto = x$txi, STAR = load_star_counts()[, rownames(samples)])
fits <- lapply(counts, \(cn) { d <- run_dds(cn, samples); list(dds = d, res = de_tables(d, genes)) })   # same exclusion of SRR21355389 as notebook 03
bind_rows(lapply(fits, \(f) bind_rows(lapply(f$res, n_deg), .id = 'contrast')), .id = 'method')

## 2 · The aging signature (21 vs 3 months)
Genes significant at padj < 0.05, split by direction; used as the signature in analyses 3–4.

In [ ]:
sig <- lapply(fits, \(f) { a <- filter(f$res$aging_21v3, padj < 0.05); list(up = a$gene_id[a$log2FoldChange > 0], down = a$gene_id[a$log2FoldChange < 0]) })
tibble(method = names(sig), up = lengths(lapply(sig, `[[`, 'up')), down = lengths(lapply(sig, `[[`, 'down')))

## 3 · Overlap of intervention DEGs with aging DEGs
`reversal` = same gene, opposite direction to aging (rejuvenating); `same` = same direction (aging-like). `fisher_p` tests enrichment of shared genes beyond chance; `lfc_spearman_agingDEG` is the log2FC correlation among aging DEGs (**negative = reversal**).

In [ ]:
ov <- bind_rows(lapply(names(fits), \(m) bind_rows(lapply(c('aDR', 'YMP'), \(n)
  overlap_with_aging(fits[[m]]$res$aging_21v3, fits[[m]]$res[[n]]) %>% mutate(intervention = n))) %>% mutate(method = m))) %>% relocate(method, intervention)
ov

In [ ]:
lfc_vs_aging_plot(fits$kallisto$res$aging_21v3, fits$kallisto$res$aDR, 'aDR vs AL, against aging (kallisto)')

In [ ]:
lfc_vs_aging_plot(fits$kallisto$res$aging_21v3, fits$kallisto$res$YMP, 'YMP vs PBS, against aging (kallisto)')

## 4 · Aging-signature score per sample
Score = mean z(aging-up genes) − mean z(aging-down genes). The aging panel is a sanity check (the signature comes from these samples); aDR and YMP are the test: a rejuvenating treatment should **lower** the score vs its control.

In [ ]:
scores <- lapply(names(fits), \(m) lapply(fits[[m]]$dds, signature_score, up = sig[[m]]$up, down = sig[[m]]$down)) %>% setNames(names(fits))
score_plot(scores$kallisto)

In [ ]:
score_tests <- bind_rows(lapply(names(scores), \(m) bind_rows(score_test(scores[[m]]$aDR, 'aDR', 'AL'), score_test(scores[[m]]$YMP, 'YMP', 'PBS')) %>% mutate(method = m))) %>% relocate(method)
score_tests

`diff` < 0 = treated lower than control (rejuvenating direction). With 3–5 mice per group the t-test p-values are rough.

## 5 · Pathways (GSEA on the DESeq2 statistic)
Hallmark (50 sets) and GO biological process (~7,500 sets); fgsea, mouse gene symbols from MSigDB.

In [ ]:
sets <- msigdb_sets()
contrasts <- c('aging_21v3', 'aDR', 'YMP')
gs_h <- lapply(fits, \(f) lapply(f$res[contrasts], run_gsea, sets = sets$hallmark))
gs_go <- lapply(fits, \(f) lapply(f$res[contrasts], run_gsea, sets = sets$gobp))
bind_rows(lapply(names(gs_h), \(m) bind_rows(lapply(names(gs_h[[m]]), \(n) tibble(method = m, contrast = n, hallmark_sig = sum(gs_h[[m]][[n]]$padj < 0.05), gobp_sig = sum(gs_go[[m]][[n]]$padj < 0.05))))))

In [ ]:
nes_tiles(gs_h$kallisto)   # blue = down, red = up; * = padj < 0.05. Opposite colours between aging and a treatment = reversal

In [ ]:
bind_rows(Hallmark = bind_rows(lapply(gs_h, gsea_concordance), .id = 'method'), GO_BP = bind_rows(lapply(gs_go, gsea_concordance), .id = 'method'), .id = 'collection')

## 6 · The paper's named gene sets in our cerebellum data
**Aging (CAS, Fig. 2B):** the 26 genes named in the figure (the full 82-gene list is in the paper's Table S1, not available here); direction is taken from our own aging data. **aDR (Fig. 6J):** the 23 genes the paper reports as aDR-regulated in ≥ 4 regions (circadian/cycling).

In [ ]:
cas <- c('Gpr17','Hsph1','Dnajb1','Chordc1','P4ha1','Ahsa1','Hist3h2a','C4b','H2-Q6','H2-Q7','Cirbp','Flt1','Tipin','Ctsz','Pcdhb14','Abca8a','Pcdhb8','Pcdhb6','Pcdhb9','H2-K1','Psmb8','Hvcn1','C1qa','Ctss','Tyrobp','Trem2')
adr23 <- c('Chordc1','Cxcl12','Fabp7','Kdr','Adipor2','Gjb6','Mertk','Cry1','Klf15','Dbp','Usp2','Etnppl','Ciart','Sult1a1','Zbtb16','Fkbp5','Sgk1','Tsc22d3','Cirbp','Rbm3','Plin4','Fmo2','Hif3a')
gene_table(fits$kallisto$res, cas) %>% arrange(padj_aging)

In [ ]:
bind_rows(lapply(fits, \(f) { t <- gene_table(f$res, adr23); tibble(genes = nrow(t), up_in_aDR = sum(t$lfc_aDR > 0), significant_aDR = sum(t$padj_aDR < 0.05, na.rm = TRUE)) }), .id = 'method')

## 7 · Save
Small summary tables for the report.

In [ ]:
dir.create(file.path(ROOT, 'results/aging_vs_rejuvenation'), showWarnings = FALSE)
out <- file.path(ROOT, 'results/aging_vs_rejuvenation')
write_csv(ov, file.path(out, 'overlap_with_aging.csv')); write_csv(score_tests, file.path(out, 'signature_score_tests.csv'))
write_csv(bind_rows(lapply(gs_h, \(g) bind_rows(g, .id = 'contrast')), .id = 'method'), file.path(out, 'gsea_hallmark.csv'))
write_csv(bind_rows(Hallmark = bind_rows(lapply(gs_h, gsea_concordance), .id = 'method'), GO_BP = bind_rows(lapply(gs_go, gsea_concordance), .id = 'method'), .id = 'collection'), file.path(out, 'gsea_concordance.csv'))
write_csv(gene_table(fits$kallisto$res, cas), file.path(out, 'paper_CAS_named_genes.csv'))
list.files(out)